# Non-Homologous TARDIS Workflow

In [1]:
from numba import config as nbconfig

nbconfig.DISABLE_JIT = False

In [2]:
import timeit

import astropy.units as u
from numpy.testing import assert_allclose

from tardis.io.configuration.config_reader import Configuration
from tardis.transport.geometry.calculate_distances import (
    calculate_distance_line,
    calculate_distance_line_nonhomologous,
)
from tardis.transport.montecarlo.configuration.constants import C_SPEED_OF_LIGHT
from tardis.transport.montecarlo.packets.radiative_packet import RPacket
from tardis.workflows.nonhomologous_tardis_workflow import (
    NonhomologousTARDISWorkflow,
)
from tardis.workflows.simple_tardis_workflow import SimpleTARDISWorkflow

/home/runner/work/tardis/tardis/tardis/__init__.py:17: UserWarning: Astropy is already imported externally. Astropy should be imported after TARDIS.
  warnings.warn(


Iterations:          0/? [00:00<?, ?it/s]

Packets:             0/? [00:00<?, ?it/s]

Initializing tabulator and plotly panel extensions for widgets to work


Auto-detected Sphinx build environment


Auto-detected Sphinx build environment


In [3]:
config = Configuration.from_yaml("../../nonhomologous_tardis_example.yml")
homologous_config = Configuration.from_yaml("../../tardis_example.yml")
config.montecarlo["enable_full_relativity"] = False
config.montecarlo["no_of_packets"] = 1
config.montecarlo["last_no_of_packets"] = 1
config.montecarlo["iterations"] = 1
homologous_config.montecarlo["enable_full_relativity"] = False
homologous_config.montecarlo["no_of_packets"] = 1
homologous_config.montecarlo["last_no_of_packets"] = 1
homologous_config.montecarlo["iterations"] = 1

In [4]:
workflow = NonhomologousTARDISWorkflow(config)
hom_workflow = SimpleTARDISWorkflow(homologous_config)

HTML(str, height=10, sizing_mode='stretch_width', styles={'border': '1px solid #ddd...})

HTML(str, height=10, sizing_mode='stretch_width', styles={'border': '1px solid #ddd...})

HTML(str, height=10, sizing_mode='stretch_width', styles={'border': '1px solid #ddd...})

In [5]:
workflow.run()

In [6]:
hom_workflow.run()

In [7]:
geometry = workflow.simulation_state.geometry.to_numba()
time_explosion = workflow.simulation_state.time_explosion.cgs.value

shell_id = 0
r_inner = geometry.r_inner[shell_id]
r_outer = geometry.r_outer[shell_id]
v_inner = geometry.v_inner[shell_id]
v_outer = geometry.v_outer[shell_id]
dvdr = geometry.velocity_gradient[shell_id]

r_test = r_inner * 1.001
mu_test = 0.5
v_test = v_inner + dvdr * (r_test - r_inner)
beta_test = v_test / C_SPEED_OF_LIGHT
doppler_factor = 1.0 - mu_test * beta_test

line_list_nu = workflow.opacity_states["opacity_state"].line_list_nu.values
line_idx = 0
nu_line_test = line_list_nu[line_idx]

comov_nu_test = nu_line_test * 1.001
nu_rest_test = comov_nu_test / doppler_factor

test_packet = RPacket(
    r=r_test,
    nu=nu_rest_test,
    mu=mu_test,
    energy=1.0,
    seed=1963,
    index=0,
)

# Double check that the calculations are not returning MISS_DISTANCE
# and that they agree
d_nonhom = calculate_distance_line_nonhomologous(
    test_packet, geometry, nu_line_test
)
d_hom = calculate_distance_line(
    test_packet,
    comov_nu_test,
    is_last_line=False,
    nu_line=nu_line_test,
    time_explosion=time_explosion,
    enable_full_relativity=False,
)
print(f"JIT Disabled: {nbconfig.DISABLE_JIT}")
print(f"\nDistance (non-homologous) : {d_nonhom:.6e} cm")
print(f"Distance (homologous)     : {d_hom:.6e} cm")
assert d_nonhom < 1e99
assert d_hom < 1e99
assert_allclose(d_nonhom, d_hom, rtol=1e-10)

# Run timeit to compare performance over many calls
N_ITER = 100000
t_nonhom = timeit.timeit(
    lambda: calculate_distance_line_nonhomologous(
        test_packet, geometry, nu_line_test
    ),
    number=N_ITER,
) * u.s
t_hom = timeit.timeit(
    lambda: calculate_distance_line(
        test_packet,
        comov_nu_test,
        False,
        nu_line_test,
        time_explosion,
        False,
    ),
    number=N_ITER,
) * u.s

print(f"\nTiming over {N_ITER} iterations:")
print(
    f"  calculate_distance_line              : {t_hom.to(u.us) / N_ITER:.3f} / call  (total {t_hom:.3f})"
)
print(
    f"  calculate_distance_line_nonhomologous: {t_nonhom.to(u.us) / N_ITER:.3f} / call  (total {t_nonhom:.3f})"
)
print(f"  Ratio (non-hom / hom)                : {t_nonhom / t_hom:.1f}")


JIT Disabled: False

Distance (non-homologous) : 3.302129e+13 cm
Distance (homologous)     : 3.302129e+13 cm



Timing over 100000 iterations:
  calculate_distance_line              : 1.508 us / call  (total 0.151 s)
  calculate_distance_line_nonhomologous: 50.096 us / call  (total 5.010 s)
  Ratio (non-hom / hom)                : 33.2
